# MTCNN

### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [14]:
!pip install mtcnn

## Part 1 - Data Preprocessing

In [15]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hungle3401/faceforensics")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'faceforensics' dataset.
Path to dataset files: /kaggle/input/faceforensics


In [16]:
# Split Videos
import os
import shutil
import random

input_root = "/kaggle/input/faceforensics/FF++"
output_root = "/content/FF++_split"

classes = ["fake", "real"]

# split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

for cls in classes:
    videos = os.listdir(os.path.join(input_root, cls))
    videos = [v for v in videos if v.endswith((".mp4", ".avi", ".mov", ".mkv"))]

    random.shuffle(videos)

    total = len(videos)
    train_end = int(train_ratio * total)
    val_end = int((train_ratio + val_ratio) * total)

    splits = {
        "train": videos[:train_end],
        "val": videos[train_end:val_end],
        "test": videos[val_end:]
    }

    for split_name, split_videos in splits.items():
        split_dir = os.path.join(output_root, split_name, cls)
        os.makedirs(split_dir, exist_ok=True)

        for v in split_videos:
            src = os.path.join(input_root, cls, v)
            dst = os.path.join(split_dir, v)
            shutil.copy(src, dst)

print("✅ Video-level split completed.")

✅ Video-level split completed.


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Step B: after processing, copy to Drive
!cp -r /content/FF++_split /content/drive/MyDrive/

^C


In [19]:
# Verify
for split in ["train", "val", "test"]:
    for cls in ["fake", "real"]:
        path = f"/content/drive/MyDrive/FF++_split/{split}/{cls}"
        print(split, cls, len(os.listdir(path)))

train fake 145
train real 140
val fake 30
val real 30
test fake 30
test real 30


In [20]:
train_videos = set(os.listdir("/content/drive/MyDrive/FF++_split/train/fake") +
                   os.listdir("/content/drive/MyDrive/FF++_split/train/real"))

val_videos = set(os.listdir("/content/drive/MyDrive/FF++_split/val/fake") +
                 os.listdir("/content/drive/MyDrive/FF++_split/val/real"))

test_videos = set(os.listdir("/content/drive/MyDrive/FF++_split/test/fake") +
                  os.listdir("/content/drive/MyDrive/FF++_split/test/real"))

print("Train ∩ Val:", len(train_videos & val_videos))
print("Train ∩ Test:", len(train_videos & test_videos))
print("Val ∩ Test:", len(val_videos & test_videos))

Train ∩ Val: 2
Train ∩ Test: 3
Val ∩ Test: 0


In [21]:
!pip install lz4

In [ ]:
import os
import cv2
import numpy as np
from mtcnn import MTCNN

detector = MTCNN()

input_root = "/content/FF++_split"
output_root = "/content/faces_mtcnn"

splits = ["train", "val", "test"]
classes = ["fake", "real"]
img_size = (224, 224)
frame_skip = 10
video_exts = (".mp4", ".avi", ".mov", ".mkv")

for split in splits:
    for cls in classes:
        input_dir = os.path.join(input_root, split, cls)
        output_dir = os.path.join(output_root, split, cls)
        os.makedirs(output_dir, exist_ok=True)

        for video_name in os.listdir(input_dir):
            if not video_name.lower().endswith(video_exts):
                continue

            video_path = os.path.join(input_dir, video_name)
            cap = cv2.VideoCapture(video_path)

            frame_id = 0
            saved_id = 0
            base_name = os.path.splitext(video_name)[0]

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_id % frame_skip == 0:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    results = detector.detect_faces(rgb)

                    if len(results) > 0:
                        # choose largest face
                        results = sorted(
                            results,
                            key=lambda x: x["box"][2] * x["box"][3],
                            reverse=True
                        )
                        x, y, w, h = results[0]["box"]

                        x = max(0, x)
                        y = max(0, y)
                        w = max(1, w)
                        h = max(1, h)

                        face = rgb[y:y+h, x:x+w]
                        if face.size > 0:
                            face = cv2.resize(face, img_size)
                            save_name = f"{base_name}_{saved_id:04d}.jpg"
                            save_path = os.path.join(output_dir, save_name)
                            cv2.imwrite(save_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                            saved_id += 1

                frame_id += 1

            cap.release()
            print(f"{split}/{cls} - {video_name}: saved {saved_id} faces")

print("Done.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step B: after processing, copy to Drive
!cp -r /content/faces_mtcnn /content/drive/MyDrive/

## Part 2 - Building the CNN

## Part 3 - Training the CNN

## Part 4 - Evaluating the Model

## Part 5 - Making a single prediction